# variableMeanSpatialNoise

Standalone notebook for the `VariableMeanSpatialNoise` protocol. Use this for real VM-noise experiments — pick a date that ran the protocol and the same pipeline (StimBlock + ResponseBlock + nearest noise chunk via `create_mea_pipeline`) applies.

Below is a synthetic demo path used to visualize what the noise stimulus looks like overlaid on the recording array. It's useful as a sanity check for geometry (canvas → MEA chip) when no real VM-noise data is available; replace the synthetic `stim_block` with a real one once you have a recording.

In [ ]:
import os
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Build a pipeline

For the synthetic VM-noise demo we just need an `AnalysisChunk` to borrow its rig geometry (`canvas_size`, `microns_per_pixel`) and its mosaic. We piggy-back on whatever experiment is first in the protocol registry. Swap `PROTOCOL_SEARCH` / `exp_name` / `datafile_name` for a real VM-noise recording when ready.

In [ ]:
PROTOCOL_SEARCH = 'AlternatingBackground'   # change to 'VariableMeanSpatialNoise' for real data

exp_search = ra.get_datasets_from_protocol_names(PROTOCOL_SEARCH)
available  = os.listdir(ra.ANALYSIS_DIR)
exp_search = exp_search.query('exp_name in @available').reset_index(drop=True)

exp_name      = exp_search['exp_name'].iloc[0]
datafile_name = exp_search.query('exp_name == @exp_name')['datafile_name'].iloc[0]

# Auto-detect ss_version + typing file (same logic as chrisMain.ipynb).
sort_dir   = os.path.join(ra.DATA_DIR, exp_name, datafile_name)
ss_version = 'kilosort2.5' if 'kilosort2.5' in os.listdir(sort_dir) else next(
    d for d in os.listdir(sort_dir) if d.startswith('kilosort'))
from retinanalysis.classes.stim import MEAStimBlock
_tmp = MEAStimBlock(exp_name, datafile_name, verbose=False)
chunk_dir   = os.path.join(ra.ANALYSIS_DIR, exp_name, _tmp.nearest_noise_chunk, ss_version)
typing_file = next(f for f in os.listdir(chunk_dir)
                   if f.endswith('.classification.txt') and not f.startswith('.'))

pipeline       = ra.create_mea_pipeline(exp_name, datafile_name,
                                        ss_version=ss_version, typing_file=typing_file)
analysis_chunk = pipeline.analysis_chunk
response_block = pipeline.resp

# Normalize cell-type labels through the canonical mapper.
response_block.df_spike_times['cell_type'] = (
    response_block.df_spike_times['cell_type'].apply(
        lambda t: ra.map_cell_type(t) or t)
)
type_counts   = response_block.df_spike_times['cell_type'].value_counts()
present_types = [t for t, n in type_counts.items() if n >= 3]
cell_types    = ra.filter_available_types(
    ['OnP', 'OffP', 'OnM', 'OffM', 'SBC', 'A1', 'BT', 'OnS', 'OffS'], present_types,
)
print(f'Using {exp_name}/{datafile_name}, chunk={analysis_chunk.chunk_name}, '
      f'cell_types={cell_types}')

## 2. Synthetic `VariableMeanSpatialNoise` overlay

This date didn't run `VariableMeanSpatialNoise`, but we can synthesize a stim_block matching the rig (`canvasSize=[800, 600]`, `micronsPerPixel=3.8`) and feed it through the same dispatcher to see what the noise stimulus would look like overlaid on the mosaic + electrodes. Switching protocols requires no extra plumbing — `regen_stimulus` and `render_displayed_canvas` route on `stim_block.protocol_name` / `stim_ds.attrs['protocol_name']`.

In [ ]:
from types import SimpleNamespace

# Use the real rig's display params so the rendered noise lives in the same
# canvas-pixel coords as the mosaic + electrodes.
canvas_size = list(analysis_chunk.canvas_size)         # [800, 600]
mu_per_pix  = float(analysis_chunk.microns_per_pixel)  # 3.8
grid_um     = 30                                       # protocol default
stixel_um   = 90                                       # protocol default → stepsPerStixel=3
steps_per_stixel = max(round(stixel_um / grid_um), 1)
grid_size_pix   = round(grid_um / mu_per_pix)          # 8
stixel_size_pix = grid_size_pix * steps_per_stixel     # 24
n_x = int(np.ceil(canvas_size[0] / stixel_size_pix) + 1)
n_y = int(np.ceil(canvas_size[1] / stixel_size_pix) + 1)

mean_intensities = [0.03, 0.3]                          # low / high mean
total_frames = 120                                      # 2 s @ 60 Hz
frames_per_switch = 60                                  # 1 s per mean (block-level=1000 ms)

epoch_params = [{
    'seed': 42,
    'numXStixels': n_x, 'numYStixels': n_y,
    'numXChecks': n_x * steps_per_stixel, 'numYChecks': n_y * steps_per_stixel,
    'stixelSize': stixel_um, 'stepsPerStixel': steps_per_stixel,
    'frameDwell': 1, 'meanSwitchInterval': 1000,
    'meanIntensities': mean_intensities,
    'totalFrames': total_frames, 'framesPerSwitch': frames_per_switch,
    'canvasSize': canvas_size, 'micronsPerPixel': mu_per_pix, 'gridSize': grid_um,
}]
synth_stim = SimpleNamespace(
    protocol_name='edu.washington.riekelab.chris.protocols.VariableMeanSpatialNoise',
    exp_name=analysis_chunk.exp_name, datafile_name='(synthetic)',
    df_epochs=pd.DataFrame({'epoch_parameters': epoch_params}),
    d_epoch_block_params={
        'preTime': 250, 'stimTime': 2000, 'tailTime': 250,
        'contrast': 1.0, 'gridSize': grid_um, 'meanSwitchInterval': 1000,
        'meanIntensities': mean_intensities,
    },
)

# Regen → render → overlay. Same three calls as for the eye-movement protocol.
ds_vm = ra.regen_stimulus(synth_stim, verbose=True)
canvas_low  = ra.render_displayed_canvas(ds_vm, epoch=0, frame=0)    # mean=0.03
canvas_high = ra.render_displayed_canvas(ds_vm, epoch=0, frame=60)   # mean=0.30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ra.plot_stim_with_mosaic(
    canvas_low, analysis_chunk, cell_types=cell_types, minimum_n=3,
    ax=axes[0], show_electrodes=True,
    electrode_kwargs=dict(s=5, c='cyan', edgecolors='black', linewidths=0.3),
    title=f'VM noise — mean={mean_intensities[0]} (low background)',
)
ra.plot_stim_with_mosaic(
    canvas_high, analysis_chunk, cell_types=cell_types, minimum_n=3,
    ax=axes[1], show_electrodes=True,
    electrode_kwargs=dict(s=5, c='red', edgecolors='black', linewidths=0.3),
    title=f'VM noise — mean={mean_intensities[1]} (high background)',
)
plt.tight_layout()
plt.show()